In [1]:

%pip install pyspark



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, FloatType, TimestampType
from pathlib import Path

spark = (
    SparkSession.builder
    .appName("RideSharingDriverPerformance")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark session started successfully")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/14 01:36:48 WARN Utils: Your hostname, Gauris-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.72 instead (on interface en0)
26/08/14 01:36:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/14 01:36:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2
Spark session started successfully


In [3]:
from pathlib import Path

cwd = Path.cwd().resolve()

possible_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent
]

PROJECT_ROOT = None

for path in possible_roots:
    if (
        (path / "data" / "drivers.csv").exists()
        and (path / "data" / "trips.csv").exists()
        and (path / "data" / "trip_logs.csv").exists()
    ):
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Make sure data/drivers.csv, "
        "data/trips.csv and data/trip_logs.csv exist."
    )

DATA_PATH = PROJECT_ROOT / "data"
OUTPUT_PATH = PROJECT_ROOT / "output"

BRONZE_PATH = OUTPUT_PATH / "bronze"
SILVER_PATH = OUTPUT_PATH / "silver"
GOLD_PATH = OUTPUT_PATH / "gold"

for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Data Path:", DATA_PATH)
print("Output Path:", OUTPUT_PATH)

Project Root: /Users/gauridargan/Downloads/Ride_Sharing_Analytics_Project_Completed
Data Path: /Users/gauridargan/Downloads/Ride_Sharing_Analytics_Project_Completed/data
Output Path: /Users/gauridargan/Downloads/Ride_Sharing_Analytics_Project_Completed/output


In [4]:
drivers_file = DATA_PATH / "drivers.csv"
trips_file = DATA_PATH / "trips.csv"
logs_file = DATA_PATH / "trip_logs.csv"

print("Drivers file exists:", drivers_file.exists())
print("Trips file exists:", trips_file.exists())
print("Trip logs file exists:", logs_file.exists())

Drivers file exists: True
Trips file exists: True
Trip logs file exists: True


In [5]:
# Read Drivers
drivers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(drivers_file))
)

# Read Trips
trips = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(trips_file))
)

# Read Trip Logs
trip_logs = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(logs_file))
)

print("All three datasets loaded successfully!")

All three datasets loaded successfully!


In [6]:
print("Drivers:", drivers.count())
print("Trips:", trips.count())
print("Trip Logs:", trip_logs.count())

Drivers: 150
Trips: 150
Trip Logs: 150


In [7]:
print("----- DRIVERS -----")
drivers.show(5, truncate=False)

print("----- TRIPS -----")
trips.show(5, truncate=False)

print("----- TRIP LOGS -----")
trip_logs.show(5, truncate=False)


----- DRIVERS -----
+---------+-------+------+------+
|driver_id|name   |city  |rating|
+---------+-------+------+------+
|1        |Rahul_1|Delhi |4.6   |
|2        |Priya_2|Mumbai|3.7   |
|3        |Rahul_3|Pune  |3.6   |
|4        |Sneha_4|Delhi |3.5   |
|5        |Priya_5|Mumbai|4.3   |
+---------+-------+------+------+
only showing top 5 rows
----- TRIPS -----
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|1      |78       |IT Park        |IT Park      |8.6        |0.0        |Cancelled  |
|2      |114      |Airport        |Mall         |17.54      |203.08     |Completed  |
|3      |132      |Railway Station|Mall         |17.27      |213.02     |Completed  |
|4      |58       |Railway Station|IT Park      |20.55      |185.6      |Completed  |
|5      |19       |IT Park    

In [8]:
# ============================================================
# BRONZE LAYER
# Store raw data in Parquet without transformations
# ============================================================

drivers.write \
    .mode("overwrite") \
    .parquet(str(BRONZE_PATH / "drivers"))

trips.write \
    .mode("overwrite") \
    .parquet(str(BRONZE_PATH / "trips"))

trip_logs.write \
    .mode("overwrite") \
    .parquet(str(BRONZE_PATH / "trip_logs"))

print("Bronze layer created successfully!")


Bronze layer created successfully!


In [9]:
print("Bronze Drivers:")
spark.read.parquet(str(BRONZE_PATH / "drivers")).show(5)

print("Bronze Trips:")
spark.read.parquet(str(BRONZE_PATH / "trips")).show(5)

print("Bronze Trip Logs:")
spark.read.parquet(str(BRONZE_PATH / "trip_logs")).show(5)


Bronze Drivers:
+---------+-------+------+------+
|driver_id|   name|  city|rating|
+---------+-------+------+------+
|        1|Rahul_1| Delhi|   4.6|
|        2|Priya_2|Mumbai|   3.7|
|        3|Rahul_3|  Pune|   3.6|
|        4|Sneha_4| Delhi|   3.5|
|        5|Priya_5|Mumbai|   4.3|
+---------+-------+------+------+
only showing top 5 rows
Bronze Trips:
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|      1|       78|        IT Park|      IT Park|        8.6|        0.0|  Cancelled|
|      2|      114|        Airport|         Mall|      17.54|     203.08|  Completed|
|      3|      132|Railway Station|         Mall|      17.27|     213.02|  Completed|
|      4|       58|Railway Station|      IT Park|      20.55|      185.6|  Completed|
|      5|       19|        IT Park|   

In [10]:
# ============================================================
# READ BRONZE DATA
# ============================================================

bronze_drivers = spark.read.parquet(
    str(BRONZE_PATH / "drivers")
)

bronze_trips = spark.read.parquet(
    str(BRONZE_PATH / "trips")
)

bronze_logs = spark.read.parquet(
    str(BRONZE_PATH / "trip_logs")
)

print("Bronze data loaded successfully!")

print("Drivers:", bronze_drivers.count())
print("Trips:", bronze_trips.count())
print("Trip Logs:", bronze_logs.count())


Bronze data loaded successfully!
Drivers: 150
Trips: 150
Trip Logs: 150


In [11]:
# ============================================================
# NULL VALUE CHECK
# ============================================================

print("----- DRIVERS NULLS -----")

bronze_drivers.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in bronze_drivers.columns
]).show()


print("----- TRIPS NULLS -----")

bronze_trips.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in bronze_trips.columns
]).show()


print("----- TRIP LOGS NULLS -----")

bronze_logs.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in bronze_logs.columns
]).show()

----- DRIVERS NULLS -----
+---------+----+----+------+
|driver_id|name|city|rating|
+---------+----+----+------+
|        0|   0|   0|     0|
+---------+----+----+------+

----- TRIPS NULLS -----
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|      0|        0|              0|            0|          0|          0|          0|
+-------+---------+---------------+-------------+-----------+-----------+-----------+

----- TRIP LOGS NULLS -----
+------+-------+----------+--------+-------------+-----------------+
|log_id|trip_id|start_time|end_time|delay_minutes|cancellation_flag|
+------+-------+----------+--------+-------------+-----------------+
|     0|      0|         0|      87|            0|                0|
+------+-------+----------+--------+-------------+-----------------+


In [12]:
# ============================================================
# CLEAN DRIVERS
# ============================================================

drivers_clean = (
    bronze_drivers

    # Remove duplicate drivers
    .dropDuplicates(["driver_id"])

    # Required fields
    .filter(F.col("driver_id").isNotNull())
    .filter(F.col("name").isNotNull())
    .filter(F.col("city").isNotNull())
    .filter(F.col("rating").isNotNull())

    # Rating must be between 0 and 5
    .filter(
        (F.col("rating") >= 0) &
        (F.col("rating") <= 5)
    )
)

print("Original drivers:", bronze_drivers.count())
print("Clean drivers:", drivers_clean.count())

drivers_clean.show(5)

Original drivers: 150
Clean drivers: 150
+---------+-------+------+------+
|driver_id|   name|  city|rating|
+---------+-------+------+------+
|        1|Rahul_1| Delhi|   4.6|
|        2|Priya_2|Mumbai|   3.7|
|        3|Rahul_3|  Pune|   3.6|
|        4|Sneha_4| Delhi|   3.5|
|        5|Priya_5|Mumbai|   4.3|
+---------+-------+------+------+
only showing top 5 rows


In [13]:
# ============================================================
# CLEAN TRIPS
# ============================================================

trips_clean = (
    bronze_trips

    # One record per trip
    .dropDuplicates(["trip_id"])

    # Required IDs
    .filter(F.col("trip_id").isNotNull())
    .filter(F.col("driver_id").isNotNull())

    # Required locations
    .filter(F.col("pickup_location").isNotNull())
    .filter(F.col("drop_location").isNotNull())

    # Required numeric values
    .filter(F.col("distance_km").isNotNull())
    .filter(F.col("fare_amount").isNotNull())

    # Invalid values removed
    .filter(F.col("distance_km") >= 0)
    .filter(F.col("fare_amount") >= 0)
)

print("Original trips:", bronze_trips.count())
print("Clean trips:", trips_clean.count())

trips_clean.show(5)


Original trips: 150
Clean trips: 150
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|      1|       78|        IT Park|      IT Park|        8.6|        0.0|  Cancelled|
|      2|      114|        Airport|         Mall|      17.54|     203.08|  Completed|
|      3|      132|Railway Station|         Mall|      17.27|     213.02|  Completed|
|      4|       58|Railway Station|      IT Park|      20.55|      185.6|  Completed|
|      5|       19|        IT Park|      IT Park|      12.47|     177.11|  Completed|
+-------+---------+---------------+-------------+-----------+-----------+-----------+
only showing top 5 rows


In [14]:
# ============================================================
# CLEAN TRIP LOGS
# ============================================================

logs_clean = (
    bronze_logs

    .dropDuplicates(["trip_id"])

    .filter(F.col("trip_id").isNotNull())

    # Convert strings into timestamps
    .withColumn(
        "start_time",
        F.to_timestamp("start_time")
    )

    .withColumn(
        "end_time",
        F.to_timestamp("end_time")
    )

    # Every trip should have a start time
    .filter(F.col("start_time").isNotNull())
)

print("Original logs:", bronze_logs.count())
print("Clean logs:", logs_clean.count())

logs_clean.show(5, truncate=False)

Original logs: 150
Clean logs: 150
+------+-------+-------------------+-------------------+-------------+-----------------+
|log_id|trip_id|start_time         |end_time           |delay_minutes|cancellation_flag|
+------+-------+-------------------+-------------------+-------------+-----------------+
|12    |12     |2025-01-06 14:05:00|NULL               |0            |1                |
|13    |13     |2025-01-01 18:16:00|2025-01-01 18:24:00|11           |0                |
|14    |14     |2025-01-04 14:23:00|2025-01-04 14:32:00|20           |0                |
|18    |18     |2025-01-07 15:21:00|NULL               |0            |1                |
|38    |38     |2025-01-04 04:48:00|2025-01-04 05:39:00|19           |0                |
+------+-------+-------------------+-------------------+-------------+-----------------+
only showing top 5 rows


In [15]:
# ============================================================
# JOIN DRIVERS + TRIPS + TRIP LOGS
# ============================================================

silver = (
    trips_clean.alias("t")

    # Join driver information
    .join(
        drivers_clean.alias("d"),
        F.col("t.driver_id") == F.col("d.driver_id"),
        "left"
    )

    # Join trip log information
    .join(
        logs_clean.alias("l"),
        F.col("t.trip_id") == F.col("l.trip_id"),
        "left"
    )

    .select(
        F.col("t.trip_id"),
        F.col("t.driver_id"),

        F.col("d.name").alias("driver_name"),
        F.col("d.city"),
        F.col("d.rating"),

        F.col("t.pickup_location"),
        F.col("t.drop_location"),
        F.col("t.distance_km"),
        F.col("t.fare_amount"),
        F.col("t.trip_status"),

        F.col("l.start_time"),
        F.col("l.end_time"),
        F.col("l.delay_minutes"),
        F.col("l.cancellation_flag")
    )
)

print("Joined Silver dataset:")
silver.show(10, truncate=False)

print("Silver records:", silver.count())

Joined Silver dataset:
+-------+---------+-----------+---------+------+---------------+-------------+-----------+-----------+-----------+-------------------+-------------------+-------------+-----------------+
|trip_id|driver_id|driver_name|city     |rating|pickup_location|drop_location|distance_km|fare_amount|trip_status|start_time         |end_time           |delay_minutes|cancellation_flag|
+-------+---------+-----------+---------+------+---------------+-------------+-----------+-----------+-----------+-------------------+-------------------+-------------+-----------------+
|1      |78       |Anjali_78  |Delhi    |5.0   |IT Park        |IT Park      |8.6        |0.0        |Cancelled  |2025-01-03 01:44:00|NULL               |0            |1                |
|2      |114      |Neha_114   |Bangalore|3.7   |Airport        |Mall         |17.54      |203.08     |Completed  |2025-01-02 04:34:00|2025-01-02 04:50:00|20           |0                |
|3      |132      |Priya_132  |Hyderabad|4

In [16]:
# ============================================================
# DERIVED COLUMNS
# ============================================================

silver = (
    silver

    # Trip duration in minutes
    .withColumn(
        "trip_duration_minutes",
        F.when(
            F.col("end_time").isNotNull(),
            (
                F.unix_timestamp("end_time")
                - F.unix_timestamp("start_time")
            ) / 60
        )
    )

    # Completed = 1, otherwise 0
    .withColumn(
        "completion_flag",
        F.when(
            F.lower(F.col("trip_status")) == "completed",
            1
        ).otherwise(0)
    )

    # Cancelled = 1 if trip status OR cancellation flag says cancelled
    .withColumn(
        "cancelled_flag",
        F.when(
            (F.lower(F.col("trip_status")) == "cancelled") |
            (F.col("cancellation_flag") == 1),
            1
        ).otherwise(0)
    )

    # Delay flag
    .withColumn(
        "delay_flag",
        F.when(
            F.coalesce(
                F.col("delay_minutes"),
                F.lit(0)
            ) > 0,
            1
        ).otherwise(0)
    )

    # Revenue only for completed rides
    .withColumn(
        "revenue",
        F.when(
            F.col("completion_flag") == 1,
            F.col("fare_amount")
        ).otherwise(0)
    )
)

silver.select(
    "trip_id",
    "driver_id",
    "trip_status",
    "start_time",
    "end_time",
    "trip_duration_minutes",
    "completion_flag",
    "cancelled_flag",
    "delay_flag",
    "revenue"
).show(10)


+-------+---------+-----------+-------------------+-------------------+---------------------+---------------+--------------+----------+-------+
|trip_id|driver_id|trip_status|         start_time|           end_time|trip_duration_minutes|completion_flag|cancelled_flag|delay_flag|revenue|
+-------+---------+-----------+-------------------+-------------------+---------------------+---------------+--------------+----------+-------+
|      1|       78|  Cancelled|2025-01-03 01:44:00|               NULL|                 NULL|              0|             1|         0|    0.0|
|      2|      114|  Completed|2025-01-02 04:34:00|2025-01-02 04:50:00|                 16.0|              1|             0|         1| 203.08|
|      3|      132|  Completed|2025-01-06 22:55:00|2025-01-06 23:29:00|                 34.0|              1|             0|         1| 213.02|
|      4|       58|  Completed|2025-01-07 22:14:00|2025-01-07 22:47:00|                 33.0|              1|             0|         1| 

In [17]:
# ============================================================
# SILVER VALIDATION
# ============================================================

silver = silver.filter(
    F.col("trip_id").isNotNull()
)

silver = silver.filter(
    F.col("driver_id").isNotNull()
)

silver = silver.filter(
    F.col("distance_km") >= 0
)

# Completed trips must have an end time
silver = silver.filter(
    ~(
        (F.col("completion_flag") == 1) &
        F.col("end_time").isNull()
    )
)

print("Final Silver records:", silver.count())


Final Silver records: 150


In [18]:
# ============================================================
# WRITE SILVER LAYER
# ============================================================

silver.write \
    .mode("overwrite") \
    .parquet(
        str(SILVER_PATH / "trips_enriched")
    )

print("Silver layer created successfully!")

Silver layer created successfully!


In [19]:
# ============================================================
# GOLD LAYER - OVERALL KPIs
# ============================================================

overall_kpi = silver.agg(
    
    # Total number of trips
    F.count("*").alias("total_trips"),

    # Completed trips
    F.sum("completion_flag").alias("completed_trips"),

    # Cancelled trips
    F.sum("cancelled_flag").alias("cancelled_trips"),

    # Total revenue from completed trips
    F.round(
        F.sum("revenue"),
        2
    ).alias("total_revenue"),

    # Average duration of completed trips
    F.round(
        F.avg(
            F.when(
                F.col("completion_flag") == 1,
                F.col("trip_duration_minutes")
            )
        ),
        2
    ).alias("avg_trip_duration_minutes"),

    # Average delay
    F.round(
        F.avg("delay_minutes"),
        2
    ).alias("avg_delay_minutes")
)

# Calculate percentages
overall_kpi = (
    overall_kpi

    .withColumn(
        "completion_rate_pct",
        F.round(
            F.col("completed_trips")
            / F.col("total_trips") * 100,
            2
        )
    )

    .withColumn(
        "cancellation_rate_pct",
        F.round(
            F.col("cancelled_trips")
            / F.col("total_trips") * 100,
            2
        )
    )
)

overall_kpi.show(truncate=False)

+-----------+---------------+---------------+-------------+-------------------------+-----------------+-------------------+---------------------+
|total_trips|completed_trips|cancelled_trips|total_revenue|avg_trip_duration_minutes|avg_delay_minutes|completion_rate_pct|cancellation_rate_pct|
+-----------+---------------+---------------+-------------+-------------------------+-----------------+-------------------+---------------------+
|150        |63             |87             |10091.08     |33.48                    |4.77             |42.0               |58.0                 |
+-----------+---------------+---------------+-------------+-------------------------+-----------------+-------------------+---------------------+



In [20]:
# ============================================================
# DRIVER PERFORMANCE
# ============================================================

driver_performance = (
    silver

    .groupBy(
        "driver_id",
        "driver_name",
        "city"
    )

    .agg(

        # Total trips handled by driver
        F.count("*").alias("total_trips"),

        # Completed trips
        F.sum(
            "completion_flag"
        ).alias("completed_trips"),

        # Cancelled trips
        F.sum(
            "cancelled_flag"
        ).alias("cancelled_trips"),

        # Average driver rating
        F.round(
            F.avg("rating"),
            2
        ).alias("average_rating"),

        # Revenue generated
        F.round(
            F.sum("revenue"),
            2
        ).alias("total_revenue"),

        # Average delay
        F.round(
            F.avg("delay_minutes"),
            2
        ).alias("average_delay_minutes")
    )
)

driver_performance.show(10, truncate=False)

+---------+-----------+---------+-----------+---------------+---------------+--------------+-------------+---------------------+
|driver_id|driver_name|city     |total_trips|completed_trips|cancelled_trips|average_rating|total_revenue|average_delay_minutes|
+---------+-----------+---------+-----------+---------------+---------------+--------------+-------------+---------------------+
|132      |Priya_132  |Hyderabad|2          |2              |0              |4.0           |387.74       |19.0                 |
|150      |Sneha_150  |Pune     |2          |1              |1              |4.8           |164.81       |9.5                  |
|91       |Rahul_91   |Pune     |3          |0              |3              |3.6           |0.0          |0.0                  |
|65       |Amit_65    |Bangalore|3          |3              |0              |3.6           |493.75       |9.0                  |
|20       |Vikas_20   |Mumbai   |3          |2              |1              |4.1           |336.6

In [21]:
# ============================================================
# DRIVER COMPLETION AND CANCELLATION RATES
# ============================================================

driver_performance = (
    driver_performance

    .withColumn(
        "completion_rate_pct",
        F.round(
            F.col("completed_trips")
            / F.col("total_trips") * 100,
            2
        )
    )

    .withColumn(
        "cancellation_rate_pct",
        F.round(
            F.col("cancelled_trips")
            / F.col("total_trips") * 100,
            2
        )
    )
)

driver_performance.show(10, truncate=False)

+---------+-----------+---------+-----------+---------------+---------------+--------------+-------------+---------------------+-------------------+---------------------+
|driver_id|driver_name|city     |total_trips|completed_trips|cancelled_trips|average_rating|total_revenue|average_delay_minutes|completion_rate_pct|cancellation_rate_pct|
+---------+-----------+---------+-----------+---------------+---------------+--------------+-------------+---------------------+-------------------+---------------------+
|132      |Priya_132  |Hyderabad|2          |2              |0              |4.0           |387.74       |19.0                 |100.0              |0.0                  |
|150      |Sneha_150  |Pune     |2          |1              |1              |4.8           |164.81       |9.5                  |50.0               |50.0                 |
|91       |Rahul_91   |Pune     |3          |0              |3              |3.6           |0.0          |0.0                  |0.0              

In [22]:
# ============================================================
# DRIVER RANKING USING WINDOW FUNCTION
# ============================================================

performance_window = Window.orderBy(
    
    # Higher completion rate is better
    F.col("completion_rate_pct").desc(),

    # Lower cancellation rate is better
    F.col("cancellation_rate_pct").asc(),

    # Higher rating is better
    F.col("average_rating").desc(),

    # Higher revenue is better
    F.col("total_revenue").desc()
)

driver_performance = driver_performance.withColumn(
    "performance_rank",
    F.row_number().over(performance_window)
)

driver_performance.orderBy(
    "performance_rank"
).show(20, truncate=False)

+---------+-----------+---------+-----------+---------------+---------------+--------------+-------------+---------------------+-------------------+---------------------+----------------+
|driver_id|driver_name|city     |total_trips|completed_trips|cancelled_trips|average_rating|total_revenue|average_delay_minutes|completion_rate_pct|cancellation_rate_pct|performance_rank|
+---------+-----------+---------+-----------+---------------+---------------+--------------+-------------+---------------------+-------------------+---------------------+----------------+
|38       |Sneha_38   |Pune     |2          |2              |0              |5.0           |223.44       |8.5                  |100.0              |0.0                  |1               |
|24       |Sneha_24   |Bangalore|1          |1              |0              |5.0           |163.49       |9.0                  |100.0              |0.0                  |2               |
|85       |Karan_85   |Mumbai   |1          |1              

26/08/14 01:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 01:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 01:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 01:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 01:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 01:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 0

In [23]:
# ============================================================
# HIGH-DEMAND PICKUP LOCATIONS
# ============================================================

pickup_demand = (
    silver

    .groupBy(
        "pickup_location"
    )

    .agg(

        # Number of trips
        F.count("*").alias("total_trips"),

        # Completed trips
        F.sum(
            "completion_flag"
        ).alias("completed_trips"),

        # Cancelled trips
        F.sum(
            "cancelled_flag"
        ).alias("cancelled_trips"),

        # Revenue
        F.round(
            F.sum("revenue"),
            2
        ).alias("total_revenue")
    )

    .orderBy(
        F.col("total_trips").desc()
    )
)

pickup_demand.show(
    truncate=False
)

+---------------+-----------+---------------+---------------+-------------+
|pickup_location|total_trips|completed_trips|cancelled_trips|total_revenue|
+---------------+-----------+---------------+---------------+-------------+
|Airport        |36         |13             |23             |2075.39      |
|IT Park        |34         |17             |17             |3191.85      |
|Railway Station|29         |14             |15             |1722.94      |
|Mall           |26         |13             |13             |2339.98      |
|City Center    |25         |6              |19             |760.92       |
+---------------+-----------+---------------+---------------+-------------+



In [24]:
# ============================================================
# DELAY ANALYSIS
# ============================================================

delay_analysis = (
    silver

    .groupBy(
        "driver_id",
        "driver_name",
        "city"
    )

    .agg(

        # Average delay
        F.round(
            F.avg("delay_minutes"),
            2
        ).alias("average_delay_minutes"),

        # Maximum delay
        F.max(
            "delay_minutes"
        ).alias("maximum_delay_minutes"),

        # Number of delayed trips
        F.sum(
            "delay_flag"
        ).alias("delayed_trips")
    )

    .orderBy(
        F.col("average_delay_minutes").desc()
    )
)

delay_analysis.show(
    20,
    truncate=False
)

+---------+-----------+---------+---------------------+---------------------+-------------+
|driver_id|driver_name|city     |average_delay_minutes|maximum_delay_minutes|delayed_trips|
+---------+-----------+---------+---------------------+---------------------+-------------+
|19       |Rahul_19   |Hyderabad|20.0                 |20                   |1            |
|99       |Rahul_99   |Delhi    |20.0                 |20                   |1            |
|143      |Anjali_143 |Mumbai   |19.0                 |19                   |1            |
|132      |Priya_132  |Hyderabad|19.0                 |20                   |2            |
|85       |Karan_85   |Mumbai   |19.0                 |19                   |1            |
|137      |Karan_137  |Delhi    |19.0                 |19                   |1            |
|111      |Karan_111  |Pune     |18.0                 |18                   |1            |
|87       |Vikas_87   |Delhi    |18.0                 |18                   |1  

In [25]:
# ============================================================
# REVENUE ANALYSIS BY CITY
# ============================================================

revenue_by_city = (
    silver

    .groupBy(
        "city"
    )

    .agg(

        F.count("*").alias("total_trips"),

        F.sum(
            "completion_flag"
        ).alias("completed_trips"),

        F.round(
            F.sum("revenue"),
            2
        ).alias("total_revenue"),

        F.round(
            F.avg(
                F.when(
                    F.col("completion_flag") == 1,
                    F.col("fare_amount")
                )
            ),
            2
        ).alias("average_completed_fare")
    )

    .orderBy(
        F.col("total_revenue").desc()
    )
)

revenue_by_city.show(
    truncate=False
)

+---------+-----------+---------------+-------------+----------------------+
|city     |total_trips|completed_trips|total_revenue|average_completed_fare|
+---------+-----------+---------------+-------------+----------------------+
|Delhi    |30         |18             |2851.14      |158.4                 |
|Bangalore|37         |15             |2535.43      |169.03                |
|Mumbai   |32         |13             |2286.38      |175.88                |
|Pune     |34         |10             |1493.44      |149.34                |
|Hyderabad|17         |7              |924.69       |132.1                 |
+---------+-----------+---------------+-------------+----------------------+



In [26]:
# ============================================================
# WRITE GOLD LAYER
# ============================================================

overall_kpi.write \
    .mode("overwrite") \
    .parquet(
        str(GOLD_PATH / "overall_kpi")
    )

driver_performance.write \
    .mode("overwrite") \
    .parquet(
        str(GOLD_PATH / "driver_performance")
    )


pickup_demand.write \
    .mode("overwrite") \
    .parquet(
        str(GOLD_PATH / "pickup_demand")
    )

delay_analysis.write \
    .mode("overwrite") \
    .parquet(
        str(GOLD_PATH / "delay_analysis")
    )

revenue_by_city.write \
    .mode("overwrite") \
    .parquet(
        str(GOLD_PATH / "revenue_by_city")
    )


print("Gold layer created successfully!")

26/08/14 01:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 01:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 01:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 01:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 01:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 01:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/14 0

Gold layer created successfully!


In [27]:
spark.stop()
